# 1. Постановка задачи

**Тема курсовой работы:** «Построение модели машинного обучения для прогнозирования страны пользователя социальной сети на основе профильных данных и интересов».

**Цель анализа:** построить и проверить модель машинного обучения, которая прогнозирует страну пользователя социальной сети по профильным данным.

**Объект исследования:** профили пользователей социальной сети.

**Предмет исследования:** закономерности связи между городом, демографическими признаками, интересами и страной пользователя.

**Входные данные модели:** `City`, `Gender`, `DOB`, `Interests` и производные признаки: возраст, возрастная группа, количество интересов, бинарные признаки интересов.

**Выход модели:** прогнозируемая страна пользователя, соответствующая целевой переменной `Country`.

**Тип задачи машинного обучения:** многоклассовая классификация.

**Основная метрика:** `macro-F1`, так как классы стран несбалансированы и важно учитывать качество по каждому классу.

**Дополнительные метрики:** `accuracy`, `weighted-F1`, `top-3 accuracy`.

`UserID` и `Name` исключаются из признаков: `UserID` является техническим идентификатором, а `Name` является персональным текстовым полем и несёт риск переобучения. Практический смысл модели состоит в восстановлении отсутствующего поля `Country`, проверке консистентности профиля и подготовке данных для географической аналитики.


# 2. Импорт библиотек и настройка окружения

В этом разделе задаются все пути проекта, фиксируется `random_state` и импортируются библиотеки, используемые в практической части. Если `CatBoost` или `SHAP` отсутствуют, notebook не прерывает выполнение: соответствующие этапы заменяются доступными средствами `scikit-learn`.


In [ ]:
from pathlib import Path
import ast
import json
import math
import re
import warnings
from datetime import date

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from IPython.display import display
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    top_k_accuracy_score,
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.0)
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

RANDOM_STATE = 42
AS_OF_DATE = pd.Timestamp('2026-05-23')
RARE_COUNTRY_THRESHOLD = 50

ROOT = Path.cwd()
DATA_PATH = ROOT / 'SocialMediaUsersDataset.csv'
FIG_DIR = ROOT / 'outputs' / 'figures'
TABLE_DIR = ROOT / 'outputs' / 'tables'
MODEL_DIR = ROOT / 'outputs' / 'models'
DOCX_PATH = ROOT / 'coursework_social_media_country_prediction.docx'
NOTEBOOK_PATH = ROOT / 'coursework_social_media_country_prediction.ipynb'

for path in [FIG_DIR, TABLE_DIR, MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('sklearn:', sklearn.__version__)
print('DATA_PATH:', DATA_PATH)


# 3. Загрузка исходных данных

Датасет содержит профили пользователей социальной сети. Каждая строка соответствует одному пользователю, а столбцы описывают идентификатор, имя, пол, дату рождения, интересы, город и страну. В курсовой работе используется файл `SocialMediaUsersDataset.csv`, содержащий 100000 строк и 7 исходных столбцов.


In [ ]:
df_raw = pd.read_csv(DATA_PATH)
display(df_raw.head())
print('Размер датасета:', df_raw.shape)
print('Столбцы:', list(df_raw.columns))
display(df_raw.dtypes.rename('dtype').to_frame())


# 4. Первичная проверка качества данных

Проверяются пропуски, полные дубликаты, уникальность `UserID`, количество уникальных значений и корректность преобразования `DOB` к дате. `Name` и `UserID` далее не используются как признаки модели.


In [ ]:
missing = df_raw.isna().sum().rename('missing_count')
duplicates_count = int(df_raw.duplicated().sum())
user_id_unique = int(df_raw['UserID'].nunique())
unique_counts = df_raw.nunique(dropna=False).rename('unique_count')
dob_parsed = pd.to_datetime(df_raw['DOB'], errors='coerce')
bad_dob_count = int(dob_parsed.isna().sum())

quality_summary = pd.DataFrame({
    'missing_count': missing,
    'unique_count': unique_counts,
})
display(quality_summary)
print('Полные дубликаты:', duplicates_count)
print('Уникальных UserID:', user_id_unique)
print('Некорректных DOB:', bad_dob_count)


**Вывод.** Пропуски отсутствуют, полных дубликатов нет, `UserID` уникален, а `DOB` корректно преобразуется в дату. Поля `UserID` и `Name` исключаются из модели, поскольку не являются обобщаемыми признаками и могут привести к переобучению.


# 5. Анализ целевой переменной `Country`

`Country` является целевой переменной. Поскольку она принимает много различных значений, задача относится к многоклассовой классификации. Распределение стран несбалансировано, поэтому основной метрикой выбрана `macro-F1`, а слишком редкие страны группируются в класс `Other`.


In [ ]:
country_counts = df_raw['Country'].value_counts()
rare_countries_le_5 = int((country_counts <= 5).sum())
rare_countries_lt_50 = int((country_counts < RARE_COUNTRY_THRESHOLD).sum())

print('Количество уникальных стран:', country_counts.shape[0])
print('Стран с количеством записей <= 5:', rare_countries_le_5)
print(f'Стран с количеством записей < {RARE_COUNTRY_THRESHOLD}:', rare_countries_lt_50)
display(country_counts.head(20).rename_axis('Country').reset_index(name='count'))

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(x=country_counts.head(20).values, y=country_counts.head(20).index, ax=ax, color='#4374a7')
ax.set_title('Распределение пользователей по странам: топ-20')
ax.set_xlabel('Количество пользователей')
ax.set_ylabel('Страна')
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig1_country_distribution.png', dpi=160, bbox_inches='tight')
plt.show()


# 6. Анализ исходных признаков

Исходные признаки имеют разную природу: `Gender` является категориальным признаком, `DOB` преобразуется в возраст, `City` является высококардинальным категориальным признаком, а `Interests` представляет собой составной список категорий.


## 6.1 Gender

Пол пользователя распределён почти равномерно: `Male: 50069`, `Female: 49931`. Этот признак сам по себе не должен определять страну, но может использоваться как дополнительный демографический сигнал.


In [ ]:
gender_counts = df_raw['Gender'].value_counts()
display(gender_counts.rename_axis('Gender').reset_index(name='count'))

fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(x=gender_counts.index, y=gender_counts.values, ax=ax, palette=['#4374a7', '#d9895b'])
ax.set_title('Распределение пользователей по полу')
ax.set_xlabel('Пол')
ax.set_ylabel('Количество пользователей')
for i, v in enumerate(gender_counts.values):
    ax.text(i, v + 500, str(v), ha='center')
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig3_gender_distribution.png', dpi=160, bbox_inches='tight')
plt.show()


## 6.2 DOB и возраст

Дата рождения используется не напрямую, а для расчёта возраста на дату `2026-05-23`. Возраст удобнее для анализа и моделирования, чем исходная календарная дата.


In [ ]:
df_age = df_raw.copy()
df_age['DOB_parsed'] = pd.to_datetime(df_age['DOB'], errors='coerce')
df_age['age'] = (
    AS_OF_DATE.year - df_age['DOB_parsed'].dt.year
    - ((AS_OF_DATE.month < df_age['DOB_parsed'].dt.month) |
       ((AS_OF_DATE.month == df_age['DOB_parsed'].dt.month) & (AS_OF_DATE.day < df_age['DOB_parsed'].dt.day))).astype(int)
)

age_stats = df_age['age'].describe().to_frame('age')
display(age_stats)
print('min:', int(df_age['age'].min()))
print('max:', int(df_age['age'].max()))
print('mean:', round(float(df_age['age'].mean()), 3))
print('median:', int(df_age['age'].median()))

fig, ax = plt.subplots(figsize=(12, 6))
sns.histplot(df_age['age'], bins=30, kde=False, ax=ax, color='#4374a7')
ax.set_title('Распределение пользователей по возрасту')
ax.set_xlabel('Возраст')
ax.set_ylabel('Количество пользователей')
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig2_age_distribution.png', dpi=160, bbox_inches='tight')
plt.show()


**Вывод.** Возраст пользователей находится в диапазоне от 21 до 72 лет, медианный возраст равен 46 годам. Возраст используется как производный числовой признак, а также как источник возрастных групп.


## 6.3 City

`City` является высококардинальным категориальным признаком. Он ожидаемо информативен для прогнозирования страны, но создаёт риск переобучения и проблему новых городов, отсутствующих в обучающей выборке.


In [ ]:
city_counts = df_raw['City'].value_counts()
print('Количество уникальных городов:', city_counts.shape[0])
print('Городов с количеством записей <= 2:', int((city_counts <= 2).sum()))
display(city_counts.head(20).rename_axis('City').reset_index(name='count'))

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(x=city_counts.head(20).values, y=city_counts.head(20).index, ax=ax, color='#52988b')
ax.set_title('Наиболее частые города пользователей: топ-20')
ax.set_xlabel('Количество пользователей')
ax.set_ylabel('Город')
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig_city_top20.png', dpi=160, bbox_inches='tight')
plt.show()


## 6.4 Interests

Поле `Interests` содержит список интересов. Повторы интересов внутри одной записи считаются артефактом и удаляются. После очистки интересы преобразуются в бинарные признаки.


In [ ]:
def parse_interests(value):
    try:
        parsed = ast.literal_eval('[' + str(value) + ']')
        if isinstance(parsed, str):
            parsed = [parsed]
    except Exception:
        parsed = [x.strip().strip("'").strip('"') for x in str(value).split(',') if x.strip()]
    return [str(x).strip() for x in parsed if str(x).strip()]

def deduplicate_preserve_order(items):
    seen = set()
    result = []
    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

interest_lists_raw = df_raw['Interests'].apply(parse_interests)
rows_with_repeated_interests = int(interest_lists_raw.apply(lambda x: len(x) != len(set(x))).sum())
interest_lists_clean = interest_lists_raw.apply(deduplicate_preserve_order)
all_interests = pd.Series([item for items in interest_lists_clean for item in items])
interest_counts = all_interests.value_counts()

print('Уникальных интересов:', interest_counts.shape[0])
print('Строк с повторами внутри Interests:', rows_with_repeated_interests)
display(interest_counts.head(10).rename_axis('Interest').reset_index(name='count'))

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(x=interest_counts.head(10).values, y=interest_counts.head(10).index, ax=ax, color='#b56f4e')
ax.set_title('Наиболее частые интересы пользователей: топ-10')
ax.set_xlabel('Количество пользователей')
ax.set_ylabel('Интерес')
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig4_top_interests.png', dpi=160, bbox_inches='tight')
plt.show()


# 7. Предобработка и feature engineering

На этом этапе создаётся рабочий датафрейм для моделирования. Исключаются `UserID` и `Name`, дата рождения преобразуется в возраст и возрастную группу, интересы очищаются от повторов и кодируются бинарными признаками, редкие страны объединяются в `Other`.


In [ ]:
def make_age_group(age):
    if age < 25:
        return '18-24'
    if age < 35:
        return '25-34'
    if age < 45:
        return '35-44'
    if age < 55:
        return '45-54'
    if age < 65:
        return '55-64'
    return '65+'

def slugify_interest(value):
    value = value.lower().replace('&', 'and')
    value = re.sub(r'[^a-z0-9]+', '_', value)
    value = re.sub(r'_+', '_', value).strip('_')
    return 'interest_' + value

model_df = df_raw.copy()
model_df['DOB_parsed'] = pd.to_datetime(model_df['DOB'], errors='coerce')
model_df['age'] = (
    AS_OF_DATE.year - model_df['DOB_parsed'].dt.year
    - ((AS_OF_DATE.month < model_df['DOB_parsed'].dt.month) |
       ((AS_OF_DATE.month == model_df['DOB_parsed'].dt.month) & (AS_OF_DATE.day < model_df['DOB_parsed'].dt.day))).astype(int)
)
model_df['age_group'] = model_df['age'].apply(make_age_group)
model_df['interests_clean'] = model_df['Interests'].apply(parse_interests).apply(deduplicate_preserve_order)
model_df['interest_count'] = model_df['interests_clean'].apply(len)

interest_to_col = {interest: slugify_interest(interest) for interest in sorted(interest_counts.index)}
for interest, col in interest_to_col.items():
    model_df[col] = model_df['interests_clean'].apply(lambda items, interest=interest: int(interest in items))

country_counts_full = model_df['Country'].value_counts()
rare_country_set = set(country_counts_full[country_counts_full < RARE_COUNTRY_THRESHOLD].index)
model_df['target_country'] = model_df['Country'].where(~model_df['Country'].isin(rare_country_set), 'Other')

binary_interest_cols = list(interest_to_col.values())
feature_cols = ['Gender', 'City', 'age', 'age_group', 'interest_count'] + binary_interest_cols
X = model_df[feature_cols].copy()
y = model_df['target_country'].copy()

print('Итоговая матрица признаков:', X.shape)
print('Количество целевых классов после группировки:', y.nunique())
display(X.head())
display(y.value_counts().head(15).rename_axis('target_country').reset_index(name='count'))


**Вывод.** `age` нужен как числовое представление даты рождения, `age_group` — как интерпретируемая дискретизация возраста, `interest_count` — как интегральный показатель заполненности профиля, бинарные признаки интересов — как представление тематического профиля. Редкие страны объединяются в `Other`, чтобы снизить шум малочисленных классов и обеспечить устойчивую стратификацию.


# 8. Разведочный анализ после формирования признаков

После feature engineering анализируются сформированные числовые признаки и связи целевой переменной с полом, возрастными группами и наиболее частыми интересами. `City` ожидаемо должен иметь наиболее сильную связь с `Country`, а интересы выступают дополнительным сигналом.


In [ ]:
print('Столбцы итогового X:')
print(list(X.columns))

numeric_summary = X[['age', 'interest_count']].describe().T
numeric_summary.to_csv(TABLE_DIR / 'table3_numeric_statistics.csv', encoding='utf-8-sig')
display(numeric_summary)

fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(data=X, x='interest_count', ax=ax, color='#4374a7')
ax.set_title('Распределение количества интересов после очистки')
ax.set_xlabel('Количество уникальных интересов')
ax.set_ylabel('Количество пользователей')
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig_interest_count_distribution.png', dpi=160, bbox_inches='tight')
plt.show()

country_gender = pd.crosstab(model_df['target_country'], model_df['Gender']).loc[y.value_counts().head(12).index]
country_gender.plot(kind='bar', stacked=True, figsize=(14, 6), color=['#d9895b', '#4374a7'])
plt.title('Зависимость целевой переменной от пола: топ-12 стран')
plt.xlabel('Страна')
plt.ylabel('Количество пользователей')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_country_by_gender.png', dpi=160, bbox_inches='tight')
plt.show()

age_group_country = pd.crosstab(model_df['age_group'], model_df['target_country'].where(model_df['target_country'].isin(y.value_counts().head(8).index), 'Other/top-rest'))
age_group_country.plot(kind='bar', stacked=True, figsize=(14, 6), colormap='tab20')
plt.title('Распределение стран по возрастным группам')
plt.xlabel('Возрастная группа')
plt.ylabel('Количество пользователей')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_country_by_age_group.png', dpi=160, bbox_inches='tight')
plt.show()

interest_country_rows = []
for interest, col in list(interest_to_col.items())[:]:
    top_country_for_interest = model_df.loc[model_df[col] == 1, 'target_country'].value_counts().head(1)
    if not top_country_for_interest.empty:
        interest_country_rows.append({
            'interest': interest,
            'top_country': top_country_for_interest.index[0],
            'count': int(top_country_for_interest.iloc[0]),
            'users_with_interest': int(model_df[col].sum()),
        })
interest_country_df = pd.DataFrame(interest_country_rows).sort_values('users_with_interest', ascending=False).head(10)
display(interest_country_df)
interest_country_df.to_csv(TABLE_DIR / 'interest_country_dependency.csv', index=False, encoding='utf-8-sig')


# 9. Подготовка train/test split

Данные делятся на обучающую и тестовую части с `test_size=0.2` и `random_state=42`. Стратификация по целевой переменной нужна, чтобы распределение стран в train и test было сопоставимым. Test-выборка не используется при подборе гиперпараметров.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train classes:', y_train.nunique())
print('y_test classes:', y_test.nunique())

split_check = pd.DataFrame({
    'train_share': y_train.value_counts(normalize=True).head(10),
    'test_share': y_test.value_counts(normalize=True).head(10),
})
display(split_check)


# 10. Базовые модели

Базовые модели нужны для проверки, имеет ли сложная модель преимущество над простыми правилами. Используются два ориентира: самый частый класс и самый частый класс внутри города.


In [ ]:
class MostFrequentByCityClassifier(BaseEstimator, ClassifierMixin):
    def fit(self, X, y):
        data = pd.DataFrame({'City': X['City'].values, 'y': pd.Series(y).values})
        self.classes_ = np.array(sorted(pd.Series(y).unique()))
        self.global_counts_ = pd.Series(y).value_counts()
        self.global_class_ = self.global_counts_.index[0]
        self.global_top3_ = list(self.global_counts_.head(3).index)
        self.city_rankings_ = {
            city: list(group['y'].value_counts().index)
            for city, group in data.groupby('City')
        }
        return self

    def predict(self, X):
        return np.array([
            self.city_rankings_.get(city, [self.global_class_])[0]
            for city in X['City'].values
        ])

    def predict_top3(self, X):
        result = []
        for city in X['City'].values:
            ranking = self.city_rankings_.get(city, []) + self.global_top3_
            unique = []
            for cls in ranking:
                if cls not in unique:
                    unique.append(cls)
                if len(unique) == 3:
                    break
            while len(unique) < 3:
                unique.append(self.global_class_)
            result.append(unique)
        return result


def top3_from_estimator(estimator, X_data, labels):
    if hasattr(estimator, 'predict_proba'):
        scores = estimator.predict_proba(X_data)
        estimator_labels = getattr(estimator, 'classes_', labels)
    elif hasattr(estimator, 'decision_function'):
        scores = estimator.decision_function(X_data)
        estimator_labels = getattr(estimator, 'classes_', labels)
    else:
        return None
    scores_df = pd.DataFrame(scores, columns=estimator_labels)
    scores_df = scores_df.reindex(columns=labels, fill_value=-np.inf)
    return scores_df.values


def evaluate_predictions(name, y_true, y_pred, top3_labels=None, scores=None, labels=None, params=''):
    if scores is not None and labels is not None:
        top3_acc = top_k_accuracy_score(y_true, scores, k=3, labels=labels)
    elif top3_labels is not None:
        top3_acc = np.mean([true in pred for true, pred in zip(y_true, top3_labels)])
    else:
        top3_acc = np.nan
    return {
        'Модель': name,
        'Гиперпараметры': params,
        'Accuracy': accuracy_score(y_true, y_pred),
        'macro-F1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted-F1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'top-3 accuracy': top3_acc,
    }

class_labels = np.array(sorted(y_train.unique()))
baseline_rows = []

most_frequent = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
most_frequent.fit(X_train, y_train)
y_pred_most = most_frequent.predict(X_test)
global_top3 = list(y_train.value_counts().head(3).index)
baseline_rows.append(evaluate_predictions(
    'Most Frequent Country', y_test, y_pred_most,
    top3_labels=[global_top3] * len(y_test),
    params='strategy=most_frequent',
))

city_baseline = MostFrequentByCityClassifier().fit(X_train, y_train)
y_pred_city = city_baseline.predict(X_test)
y_top3_city = city_baseline.predict_top3(X_test)
baseline_rows.append(evaluate_predictions(
    'Most Frequent Country by City', y_test, y_pred_city,
    top3_labels=y_top3_city,
    params='city majority on train',
))

baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df)


# 11. Подготовка пайплайнов моделей

Для предотвращения утечки данных все обучаемые преобразования признаков находятся внутри `Pipeline` и `ColumnTransformer`. `OneHotEncoder(handle_unknown='ignore')` позволяет корректно обрабатывать новые города и категории, которые отсутствовали в обучающей части.


In [ ]:
numeric_features = ['age', 'interest_count']
categorical_features = ['Gender', 'City', 'age_group']

preprocess_linear = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_features),
        ('interest', 'passthrough', binary_interest_cols),
    ],
    sparse_threshold=0.3,
)

preprocess_nb = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_features),
        ('interest', 'passthrough', binary_interest_cols),
    ],
    sparse_threshold=0.3,
)

print('Числовые признаки:', numeric_features)
print('Категориальные признаки:', categorical_features)
print('Бинарных признаков интересов:', len(binary_interest_cols))


# 12. Формулирование гипотез и обучение моделей

Для проверки качества прогнозирования страны пользователя сформулированы пять экспериментальных гипотез. Каждая гипотеза соответствует отдельному классу алгоритмов и проверяется на единой схеме подготовки данных, чтобы сравнение моделей было сопоставимым.

**Гипотеза 1.** `Logistic Regression` может дать устойчивое качество на разреженных признаках после кодирования категорий. Модель использует город, пол, возрастную группу, количество интересов и бинарные признаки интересов. Ожидаемое преимущество состоит в устойчивости линейной модели на one-hot представлении, а основное ограничение связано с невозможностью напрямую учитывать сложные нелинейные взаимодействия между городом, возрастом и интересами. В качестве основных гиперпараметров рассматриваются сила регуляризации `C` и тип регуляризации.

**Гипотеза 2.** `Linear SVM` или `SGDClassifier` может быть эффективен для многоклассовой классификации при большом количестве разреженных признаков. Такая модель подходит для данных после one-hot encoding, поскольку вычислительно эффективна на больших разреженных матрицах. Преимущество заключается в скорости обучения и способности строить разделяющие гиперплоскости для большого числа классов. Ограничение состоит в чувствительности к регуляризации и масштабу числовых признаков. В эксперименте проверяется `SGDClassifier` с подбором `alpha` и функции потерь.

**Гипотеза 3.** `Complement Naive Bayes` может быть полезен как простая вероятностная модель для бинарных признаков интересов и категориальных индикаторов. Алгоритм особенно уместен как базовая вероятностная модель для разреженных неотрицательных признаков. Его преимущество состоит в высокой скорости обучения и устойчивости при большом числе категориальных индикаторов. Ограничение связано с предположением условной независимости признаков, которое для профилей пользователей выполняется не полностью. Основной гиперпараметр модели — сглаживание `alpha`.

**Гипотеза 4.** `Random Forest` или `ExtraTreesClassifier` может выявлять нелинейные зависимости между профильными признаками и страной пользователя. Деревья решений способны учитывать комбинации города, возрастной группы, пола и интересов без явного задания взаимодействий. Потенциальное преимущество состоит в моделировании нелинейных правил, а ограничение — в увеличении вычислительной сложности и возможном ухудшении качества на большом количестве разреженных one-hot признаков. В эксперименте используется `ExtraTreesClassifier` с подбором числа деревьев, глубины и минимального числа объектов в листе.

**Гипотеза 5.** `CatBoostClassifier` может быть наиболее подходящим алгоритмом для задачи, поскольку исходные данные содержат категориальные признаки высокой мощности, прежде всего `City`. Алгоритм умеет работать с категориальными признаками и снижает потребность в ручном one-hot encoding. Его ожидаемое преимущество — более корректная обработка высококардинальных категорий. Ограничение состоит в зависимости от внешней библиотеки CatBoost и более высокой вычислительной стоимости. В текущем окружении библиотека CatBoost недоступна, поэтому практическая часть выполнена средствами scikit-learn, а гипотеза CatBoost зафиксирована как перспективное направление развития эксперимента.

Для ограничения времени поиска гиперпараметров используется стратифицированная подвыборка обучающей части. После выбора параметров каждая модель переобучается на всей train-выборке. Это сохраняет этап подбора гиперпараметров и не использует test-выборку при выборе модели.



In [ ]:
SEARCH_SIZE = 30000
if len(X_train) > SEARCH_SIZE:
    X_search, _, y_search, _ = train_test_split(
        X_train, y_train,
        train_size=SEARCH_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_train,
    )
else:
    X_search, y_search = X_train, y_train

print('Размер подвыборки для GridSearchCV/RandomizedSearchCV:', X_search.shape)

model_search_specs = []

log_reg_pipe = Pipeline([
    ('preprocess', preprocess_linear),
    ('model', LogisticRegression(
        solver='saga',
        max_iter=120,
        tol=0.01,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
])
model_search_specs.append((
    'Logistic Regression',
    GridSearchCV(
        log_reg_pipe,
        param_grid={'model__C': [0.5, 1.0]},
        scoring='f1_macro',
        cv=3,
        n_jobs=-1,
        refit=True,
    )
))

sgd_pipe = Pipeline([
    ('preprocess', preprocess_linear),
    ('model', SGDClassifier(
        loss='log_loss',
        penalty='l2',
        max_iter=1000,
        early_stopping=True,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])
model_search_specs.append((
    'SGDClassifier',
    GridSearchCV(
        sgd_pipe,
        param_grid={'model__alpha': [1e-4, 1e-5]},
        scoring='f1_macro',
        cv=3,
        n_jobs=-1,
        refit=True,
    )
))

nb_pipe = Pipeline([
    ('preprocess', preprocess_nb),
    ('model', ComplementNB()),
])
model_search_specs.append((
    'Complement Naive Bayes',
    GridSearchCV(
        nb_pipe,
        param_grid={'model__alpha': [0.1, 0.5, 1.0]},
        scoring='f1_macro',
        cv=3,
        n_jobs=-1,
        refit=True,
    )
))

extra_pipe = Pipeline([
    ('preprocess', preprocess_nb),
    ('model', ExtraTreesClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight=None,
    )),
])
model_search_specs.append((
    'ExtraTreesClassifier',
    RandomizedSearchCV(
        extra_pipe,
        param_distributions={
            'model__n_estimators': [40, 60],
            'model__max_depth': [20, None],
            'model__min_samples_leaf': [1, 2],
        },
        n_iter=3,
        scoring='f1_macro',
        cv=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        refit=True,
    )
))

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except Exception as exc:
    CATBOOST_AVAILABLE = False
    print('CatBoost недоступен, этап CatBoost пропущен:', type(exc).__name__)

trained_models = []
search_rows = []

for model_name, search in model_search_specs:
    print(f'Обучение и подбор гиперпараметров: {model_name}')
    search.fit(X_search, y_search)
    best_estimator = clone(search.best_estimator_)
    best_estimator.fit(X_train, y_train)
    trained_models.append((model_name, best_estimator, search.best_params_))
    search_rows.append({
        'model': model_name,
        'best_params': json.dumps(search.best_params_, ensure_ascii=False),
        'best_cv_macro_f1': search.best_score_,
    })
    print(model_name, search.best_params_, 'cv macro-F1:', round(search.best_score_, 4))

search_results_df = pd.DataFrame(search_rows)
search_results_df.to_csv(TABLE_DIR / 'hyperparameter_search_sklearn.csv', index=False, encoding='utf-8-sig')
display(search_results_df)


# 13. Оценка моделей на test-выборке

На test-выборке фиксируются финальные метрики. Лучшая модель выбирается по `macro-F1`, так как эта метрика учитывает качество по каждому классу и не позволяет скрыть низкое качество на редких странах за счёт частых классов.


In [ ]:
result_rows = baseline_rows.copy()

for model_name, estimator, best_params in trained_models:
    y_pred = estimator.predict(X_test)
    scores = top3_from_estimator(estimator, X_test, class_labels)
    result_rows.append(evaluate_predictions(
        model_name,
        y_test,
        y_pred,
        scores=scores,
        labels=class_labels,
        params=json.dumps(best_params, ensure_ascii=False),
    ))

comparison_df = pd.DataFrame(result_rows)
for col in ['Accuracy', 'macro-F1', 'weighted-F1', 'top-3 accuracy']:
    comparison_df[col] = comparison_df[col].astype(float)

# Если несколько моделей имеют одинаковую macro-F1 после округления до 3 знаков,
# предпочтение отдаётся полноценной ML-модели, а не базовому частотному правилу.
baseline_names = {'Most Frequent Country', 'Most Frequent Country by City'}
comparison_df['is_baseline'] = comparison_df['Модель'].isin(baseline_names)
comparison_df['macro_f1_rank'] = comparison_df['macro-F1'].round(3)
comparison_df = (
    comparison_df
    .sort_values(['macro_f1_rank', 'is_baseline', 'Accuracy'], ascending=[False, True, False])
    .drop(columns=['macro_f1_rank', 'is_baseline'])
    .reset_index(drop=True)
)

comparison_rounded = comparison_df.copy()
for col in ['Accuracy', 'macro-F1', 'weighted-F1', 'top-3 accuracy']:
    comparison_rounded[col] = comparison_rounded[col].round(4)

display(comparison_rounded)
comparison_rounded.to_csv(TABLE_DIR / 'model_comparison.csv', index=False, encoding='utf-8-sig')
comparison_rounded.to_csv(TABLE_DIR / 'table5_model_comparison.csv', index=False, encoding='utf-8-sig')

best_model_name = comparison_df.iloc[0]['Модель']
best_model_metrics = comparison_rounded.iloc[0].to_dict()
print('Лучшая модель:', best_model_name)
print(best_model_metrics)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=comparison_rounded, y='Модель', x='macro-F1', ax=ax, color='#52988b')
ax.set_title('Сравнение качества моделей по macro-F1')
ax.set_xlabel('macro-F1')
ax.set_ylabel('Модель')
for i, value in enumerate(comparison_rounded['macro-F1']):
    ax.text(value + 0.005, i, f'{value:.4f}', va='center')
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig5_model_comparison.png', dpi=160, bbox_inches='tight')
plt.show()

**Вывод.** Выбор модели выполняется по `macro-F1`. Если базовое правило и ML-модель дают одинаковую `macro-F1` после округления до 3 знаков, для дальнейшей интерпретации и сохранения выбирается ML-модель, поскольку цель курсовой работы состоит в построении модели машинного обучения. Базовое правило по городу остаётся сильным ориентиром качества и показывает, насколько информативен признак `City`.

# 14. Матрица ошибок лучшей модели

Из-за большого числа целевых классов полная матрица ошибок плохо читается. Поэтому строится матрица для top-15 наиболее частых стран и агрегированного класса остальных стран.


In [ ]:
model_lookup = {'Most Frequent Country': most_frequent, 'Most Frequent Country by City': city_baseline}
model_lookup.update({name: estimator for name, estimator, params in trained_models})
best_estimator = model_lookup[best_model_name]
best_y_pred = best_estimator.predict(X_test)

major_classes = list(y_test.value_counts().head(15).index)
if 'Other' not in major_classes and 'Other' in set(y_test):
    major_classes.append('Other')

def collapse_class(value):
    return value if value in major_classes else 'Прочие'

cm_y_true = pd.Series(y_test).map(collapse_class)
cm_y_pred = pd.Series(best_y_pred).map(collapse_class)
cm_labels = major_classes + ['Прочие']
cm_labels = [x for x in cm_labels if x in set(cm_y_true) or x in set(cm_y_pred)]
cm = confusion_matrix(cm_y_true, cm_y_pred, labels=cm_labels, normalize='true')

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(cm, xticklabels=cm_labels, yticklabels=cm_labels, cmap='Blues', annot=False, ax=ax)
ax.set_title('Матрица ошибок лучшей модели: top-классы')
ax.set_xlabel('Прогноз')
ax.set_ylabel('Истинный класс')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig6_confusion_matrix.png', dpi=160, bbox_inches='tight')
plt.show()

report = classification_report(y_test, best_y_pred, zero_division=0, output_dict=True)
report_df = pd.DataFrame(report).T
report_df.to_csv(TABLE_DIR / 'classification_report_best_model.csv', encoding='utf-8-sig')
display(report_df.tail())


**Вывод.** Ошибки связаны прежде всего с неоднозначными или редкими городами. Для городов, которые хорошо представлены в train, прогноз обычно устойчив. Для новых или малочисленных городов модель вынуждена опираться на более слабые признаки: возрастную группу, пол и интересы.


# 15. Интерпретация лучшей модели

Используется доступный вариант интерпретации. Если модель имеет `feature_importances_`, выводятся важные признаки. Если модель линейная, анализируются коэффициенты. Дополнительно применяется `permutation_importance` на ограниченной подвыборке test, чтобы оценить вклад групп признаков без чрезмерной вычислительной нагрузки. Если `SHAP` недоступен, выполнение не прерывается.


In [ ]:
# 1. Прямая важность признаков для моделей, которые её поддерживают.
importance_df = None

if best_model_name in ['Logistic Regression', 'SGDClassifier', 'Complement Naive Bayes', 'ExtraTreesClassifier']:
    try:
        preprocess = best_estimator.named_steps['preprocess']
        feature_names = preprocess.get_feature_names_out()
        model = best_estimator.named_steps['model']
        if hasattr(model, 'feature_importances_'):
            values = model.feature_importances_
            importance_df = pd.DataFrame({'feature': feature_names, 'importance': values})
        elif hasattr(model, 'coef_'):
            values = np.abs(model.coef_).mean(axis=0)
            importance_df = pd.DataFrame({'feature': feature_names, 'importance': values})
        elif hasattr(model, 'feature_log_prob_'):
            values = np.abs(model.feature_log_prob_).mean(axis=0)
            importance_df = pd.DataFrame({'feature': feature_names, 'importance': values})
        if importance_df is not None:
            importance_df = importance_df.sort_values('importance', ascending=False).head(20)
    except Exception as exc:
        print('Прямая важность признаков недоступна:', type(exc).__name__, exc)

# 2. Реальная permutation importance по группам исходных признаков.
# Используется подвыборка test, чтобы расчёт был воспроизводимым и не перегружал notebook.
PERMUTATION_SAMPLE_SIZE = 5000
if len(X_test) > PERMUTATION_SAMPLE_SIZE:
    X_perm_sample, _, y_perm_sample, _ = train_test_split(
        X_test,
        y_test,
        train_size=PERMUTATION_SAMPLE_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_test,
    )
else:
    X_perm_sample = X_test.copy()
    y_perm_sample = y_test.copy()

base_perm_pred = best_estimator.predict(X_perm_sample)
base_perm_macro_f1 = f1_score(y_perm_sample, base_perm_pred, average='macro', zero_division=0)

rng = np.random.default_rng(RANDOM_STATE)
permutation_groups = {
    'City': ['City'],
    'Gender': ['Gender'],
    'age': ['age'],
    'age_group': ['age_group'],
    'interest_count': ['interest_count'],
    'binary_interests': binary_interest_cols,
}

permutation_rows = []
for group_name, columns in permutation_groups.items():
    X_permuted = X_perm_sample.copy()
    for col in columns:
        X_permuted[col] = rng.permutation(X_permuted[col].to_numpy())
    perm_pred = best_estimator.predict(X_permuted)
    perm_macro_f1 = f1_score(y_perm_sample, perm_pred, average='macro', zero_division=0)
    permutation_rows.append({
        'group': group_name,
        'base_macro_f1': base_perm_macro_f1,
        'macro_f1_after_permutation': perm_macro_f1,
        'macro_f1_drop': base_perm_macro_f1 - perm_macro_f1,
    })

permutation_group_df = pd.DataFrame(permutation_rows).sort_values('macro_f1_drop', ascending=False)
permutation_group_df.to_csv(TABLE_DIR / 'permutation_importance.csv', index=False, encoding='utf-8-sig')
display(permutation_group_df)

# Для рисунка 7 используется именно реальная permutation importance по группам признаков.
fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(data=permutation_group_df, x='macro_f1_drop', y='group', ax=ax, color='#4374a7')
ax.set_title('Permutation importance групп признаков')
ax.set_xlabel('Падение macro-F1 после перестановки')
ax.set_ylabel('Группа признаков')
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig7_feature_importance.png', dpi=160, bbox_inches='tight')
plt.show()

if importance_df is not None:
    importance_df.to_csv(TABLE_DIR / 'feature_importance.csv', index=False, encoding='utf-8-sig')
    display(importance_df)
else:
    pd.DataFrame({'note': ['Прямая важность отдельных признаков для выбранной модели недоступна']}).to_csv(
        TABLE_DIR / 'feature_importance.csv', index=False, encoding='utf-8-sig'
    )

try:
    import shap
    print('SHAP установлен, но для данной многоклассовой задачи с большим числом классов он не запускается по умолчанию, чтобы не перегружать вычисления.')
except Exception as exc:
    print('SHAP недоступен, используется permutation importance:', type(exc).__name__)

**Вывод.** Интерпретация выполнена через реальную permutation importance по группам признаков: значения получены перестановкой соответствующих столбцов на test-подвыборке и повторным расчётом `macro-F1`. Наиболее значимым признаком ожидаемо является `City`. Интересы, возрастная группа и пол дают дополнительный сигнал, но не определяют страну однозначно. Ограничение интерпретации состоит в том, что сильный вклад города может маскировать более слабые эффекты остальных признаков.

# 16. Сохранение артефактов

Сохраняются лучшая модель, список признаков, таблица сравнения моделей и графики. Эти файлы используются в Word-документе курсовой работы.


In [ ]:
best_model_path = MODEL_DIR / 'best_model.joblib'
joblib.dump({
    'model_name': best_model_name,
    'model': best_estimator,
    'feature_columns': feature_cols,
    'binary_interest_columns': binary_interest_cols,
    'class_labels': class_labels,
    'metrics': best_model_metrics,
}, best_model_path)

feature_info = {
    'feature_columns': feature_cols,
    'binary_interest_columns': binary_interest_cols,
    'target': 'target_country',
    'rare_country_threshold': RARE_COUNTRY_THRESHOLD,
    'random_state': RANDOM_STATE,
}
(TABLE_DIR / 'feature_info.json').write_text(json.dumps(feature_info, ensure_ascii=False, indent=2), encoding='utf-8')

print('Модель:', best_model_path)
print('Таблица результатов:', TABLE_DIR / 'model_comparison.csv')
print('Графики:', FIG_DIR)
print('Notebook:', NOTEBOOK_PATH)
print('DOCX:', DOCX_PATH)


# 17. Итоговые выводы

Итоговые выводы формируются на основе реальных метрик, полученных на test-выборке. Эти же таблицы, графики и метрики используются при формировании `.docx` курсовой работы.


In [ ]:
summary_text = f"""
Лучшая модель: {best_model_name}
Accuracy: {best_model_metrics['Accuracy']:.4f}
macro-F1: {best_model_metrics['macro-F1']:.4f}
weighted-F1: {best_model_metrics['weighted-F1']:.4f}
top-3 accuracy: {best_model_metrics['top-3 accuracy']:.4f}

Наиболее значимый признак по смыслу задачи — City. Интересы, возрастная группа и пол используются как дополнительные сигналы.
Notebook воспроизводит практическую часть курсовой работы: загрузку данных, EDA, feature engineering, обучение моделей, оценку качества, интерпретацию и сохранение артефактов.
"""
print(summary_text)

# Сохраняем контекст для сборки docx тем же набором результатов.
table1 = pd.DataFrame([
    {'Столбец': 'UserID', 'Тип': 'идентификатор', 'Описание': 'Уникальный идентификатор пользователя', 'Роль': 'исключается'},
    {'Столбец': 'Name', 'Тип': 'текстовый', 'Описание': 'Имя пользователя', 'Роль': 'исключается'},
    {'Столбец': 'Gender', 'Тип': 'категориальный', 'Описание': 'Пол пользователя', 'Роль': 'входной признак'},
    {'Столбец': 'DOB', 'Тип': 'дата', 'Описание': 'Дата рождения', 'Роль': 'источник возраста'},
    {'Столбец': 'Interests', 'Тип': 'составной список', 'Описание': 'Интересы пользователя', 'Роль': 'входной признак'},
    {'Столбец': 'City', 'Тип': 'категориальный', 'Описание': 'Город пользователя', 'Роль': 'входной признак'},
    {'Столбец': 'Country', 'Тип': 'категориальный', 'Описание': 'Страна пользователя', 'Роль': 'целевая переменная'},
])
table2 = pd.DataFrame([
    {'Признак': 'City', 'Использование': 'да', 'Преобразование': 'OneHotEncoder(handle_unknown="ignore")', 'Причина': 'основной географический сигнал'},
    {'Признак': 'Gender', 'Использование': 'да', 'Преобразование': 'OneHotEncoder', 'Причина': 'дополнительный демографический признак'},
    {'Признак': 'DOB', 'Использование': 'да', 'Преобразование': 'age и age_group', 'Причина': 'возраст интерпретируемее даты'},
    {'Признак': 'Interests', 'Использование': 'да', 'Преобразование': 'очистка, binary indicators, interest_count', 'Причина': 'тематический профиль'},
    {'Признак': 'UserID', 'Использование': 'нет', 'Преобразование': 'исключение', 'Причина': 'технический идентификатор'},
    {'Признак': 'Name', 'Использование': 'нет', 'Преобразование': 'исключение', 'Причина': 'персональный текстовый признак'},
])
table3 = numeric_summary.reset_index().rename(columns={'index': 'Признак'}).round(3)
table4 = pd.DataFrame([
    {'Гипотеза': 'Logistic Regression', 'Гиперпараметры': 'C: 0.5, 1.0', 'Преимущества': 'устойчива на разреженных признаках', 'Ограничения': 'итерационная оптимизация'},
    {'Гипотеза': 'SGDClassifier', 'Гиперпараметры': 'alpha: 1e-4, 1e-5', 'Преимущества': 'эффективен для больших разреженных матриц', 'Ограничения': 'чувствителен к регуляризации'},
    {'Гипотеза': 'Complement Naive Bayes', 'Гиперпараметры': 'alpha: 0.1, 0.5, 1.0', 'Преимущества': 'быстрая вероятностная модель', 'Ограничения': 'предположение независимости признаков'},
    {'Гипотеза': 'ExtraTreesClassifier', 'Гиперпараметры': 'n_estimators, max_depth, min_samples_leaf', 'Преимущества': 'нелинейные зависимости', 'Ограничения': 'тяжелее на one-hot признаках'},
    {'Гипотеза': 'CatBoostClassifier', 'Гиперпараметры': 'depth, learning_rate, iterations', 'Преимущества': 'категориальные признаки', 'Ограничения': 'библиотека недоступна в окружении'},
])
table6 = pd.DataFrame([
    {'Показатель': 'Лучшая модель', 'Значение': best_model_name},
    {'Показатель': 'Размер обучающей выборки', 'Значение': len(X_train)},
    {'Показатель': 'Размер тестовой выборки', 'Значение': len(X_test)},
    {'Показатель': 'Количество целевых классов после группировки', 'Значение': y.nunique()},
    {'Показатель': 'Основная метрика macro-F1', 'Значение': best_model_metrics['macro-F1']},
    {'Показатель': 'Accuracy', 'Значение': best_model_metrics['Accuracy']},
    {'Показатель': 'weighted-F1', 'Значение': best_model_metrics['weighted-F1']},
    {'Показатель': 'top-3 accuracy', 'Значение': best_model_metrics['top-3 accuracy']},
    {'Показатель': 'Файл модели', 'Значение': str(best_model_path)},
])

for idx, table in enumerate([table1, table2, table3, table4, comparison_rounded, table6], start=1):
    table.to_csv(TABLE_DIR / f'table{idx}_notebook.csv', index=False, encoding='utf-8-sig')

docx_context = {
    'stats': {
        'rows': int(df_raw.shape[0]),
        'cols': int(df_raw.shape[1]),
        'missing_total': int(df_raw.isna().sum().sum()),
        'duplicate_full_rows': int(df_raw.duplicated().sum()),
        'user_id_unique': int(df_raw['UserID'].nunique()),
        'country_classes_raw': int(country_counts.shape[0]),
        'country_classes_grouped': int(y.nunique()),
        'countries_le_5': int((country_counts <= 5).sum()),
        'city_unique': int(city_counts.shape[0]),
        'interest_categories': int(interest_counts.shape[0]),
        'duplicate_interest_rows': int(rows_with_repeated_interests),
        'age_min': int(model_df['age'].min()),
        'age_max': int(model_df['age'].max()),
        'age_median': int(model_df['age'].median()),
        'male_count': int(gender_counts.get('Male', 0)),
        'female_count': int(gender_counts.get('Female', 0)),
        'train_size': int(len(X_train)),
        'test_size': int(len(X_test)),
        'validation_size': int(len(X_search)),
    },
    'table1': table1.to_dict('records'),
    'table2': table2.to_dict('records'),
    'table3': table3.to_dict('records'),
    'table4': table4.to_dict('records'),
    'table5': comparison_rounded.to_dict('records'),
    'table6': table6.to_dict('records'),
    'best_name': best_model_name,
    'best_result': best_model_metrics,
    'importance_rows': permutation_group_df.rename(columns={'group': 'Группа признаков', 'macro_f1_after_permutation': 'macro-F1 после перестановки', 'macro_f1_drop': 'Падение macro-F1'}).round(6).to_dict('records'),
    'model_path': str(best_model_path),
    'top_countries': list(country_counts.head(10).items()),
}
(ROOT / 'outputs' / 'run_summary.json').write_text(json.dumps(docx_context, ensure_ascii=False, indent=2), encoding='utf-8')

# Рисунок архитектуры берётся из существующего генератора, если он уже есть; иначе строится простой вариант.
try:
    import importlib.util
    spec = importlib.util.spec_from_file_location('coursework_generator', ROOT / 'scripts' / 'generate_coursework.py')
    generator = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(generator)
    generator.draw_architecture(FIG_DIR / 'fig8_architecture.png')
    generator.build_docx(docx_context)
    print('DOCX пересобран на основе результатов notebook:', DOCX_PATH)
except Exception as exc:
    print('DOCX не был пересобран автоматически:', type(exc).__name__, exc)
